# GF3 Audio Modem – File **Encoder** (Stages 0 → Binary)
Prepared 2025-06-07

**Purpose**  
This notebook converts an input file (`.txt`, `.wav`, or `.tiff`) into a binary bit‑stream ready for QPSK mapping, following the project framing rules:

```
<filename utf‑8> 0x00  <filesize utf‑8> 0x00  <raw file bytes>
```
After building the byte stream we convert it to `uint8` and finally to a NumPy array of bits (little‑endian within each byte).

In [1]:
import numpy as np
from pathlib import Path
from typing import Tuple
from PIL import Image

## Stage 1 – Load file and sanity checks

In [2]:
def load_file(path: str) -> Tuple[bytes, str]:
    """Return raw bytes and filename (without folders)."""
    p = Path(path).expanduser().resolve()
    if not p.exists():
        raise FileNotFoundError(p)
    if p.suffix.lower() not in {'.txt', '.wav', '.tiff', '.tif'}:
        raise ValueError('Unsupported extension')
    data = p.read_bytes()
    return data, p.name

# Example (uncomment to test)
# raw, name = load_file('example.txt')

## Stage 2 – Build header (filename and size)

In [3]:
def build_header(filename: str, nbytes: int) -> bytes:
    filename_b = filename.encode('utf-8') + b'\0'
    size_b = str(nbytes).encode('utf-8') + b'\0'
    return filename_b + size_b


## Stage 3 – Assemble full byte sequence

In [4]:
def assemble_stream(path: str) -> bytes:
    data, name = load_file(path)
    header = build_header(name, len(data))
    return header + data


## Stage 4 – Convert to `uint8` numpy array

In [5]:
def bytes_to_uint8(byte_stream: bytes) -> np.ndarray:
    return np.frombuffer(byte_stream, dtype=np.uint8)


## Stage 5 – Convert `uint8` to binary bit‑array

In [6]:
def uint8_to_bits(uint8_arr: np.ndarray) -> np.ndarray:
    """Return 1‑D array of bits (0/1), MSB first per byte."""
    return np.unpackbits(uint8_arr, bitorder='big')


## Stage 6 – End‑to‑end convenience wrapper

In [7]:
def encode_file_to_bits(path: str) -> np.ndarray:
    stream = assemble_stream(path)
    u8 = bytes_to_uint8(stream)
    bits = uint8_to_bits(u8)
    return bits


## Demo

In [13]:
if True:  # set to True and edit path to test interactively
    bits = encode_file_to_bits('files/example.tiff')
    print(f'Encoded {bits.size} bits, first 64 bits:\n', bits[:64])

Encoded 41872120 bits, first 64 bits:
 [0 1 1 0 0 1 0 1 0 1 1 1 1 0 0 0 0 1 1 0 0 0 0 1 0 1 1 0 1 1 0 1 0 1 1 1 0
 0 0 0 0 1 1 0 1 1 0 0 0 1 1 0 0 1 0 1 0 0 1 0 1 1 1 0]


## Validation

In [14]:
byte_array = np.packbits(bits)

In [15]:
file_name_terminate = np.where(byte_array == 0)[0][0]
file_name = byte_array[:file_name_terminate].tobytes().decode("utf-8")
file_name

'example.tiff'

In [16]:
file_size_terminate = np.where(byte_array == 0)[0][1]
file_size = int(byte_array[file_name_terminate + 1:file_size_terminate].tobytes().decode("utf-8"))
file_size

5233994

In [17]:
file_content = byte_array[(file_size_terminate + 1):(file_size_terminate + int(file_size) + 1)]
with open("file_1.tiff", "wb") as f:
    f.write(file_content.tobytes())